In [1]:
import pandas as pd


In [4]:
filepath = r"C:\Users\nihar\Downloads\Extensive_A_Z_medicines_dataset_of_India.csv"
df = pd.read_csv(filepath)

In [9]:
df.shape

(256476, 24)

In [8]:
df.columns

Index(['id', 'name', 'price(₹)', 'Is_discontinued', 'manufacturer_name',
       'type', 'pack_size_label', 'short_composition1', 'short_composition2',
       'substitute0', 'substitute1', 'substitute2', 'substitute3',
       'substitute4', 'Consolidated_Side_Effects', 'use0', 'use1', 'use2',
       'use3', 'use4', 'Chemical Class', 'Habit Forming', 'Therapeutic Class',
       'Action Class'],
      dtype='object')

In [7]:
df.head()

,id,name,price(₹),Is_discontinued,manufacturer_name,type,pack_size_label,short_composition1,short_composition2,substitute0,...,Consolidated_Side_Effects,use0,use1,use2,use3,use4,Chemical Class,Habit Forming,Therapeutic Class,Action Class
0,1,Augmentin 625 Duo Tablet,223.42,False,Glaxo SmithKline Pharmaceuticals Ltd,allopathy,strip of 10 tablets,Amoxycillin (500mg),Clavulanic Acid (125mg),Penciclav 500 mg/125 mg Tablet,...,"Vomiting, Nausea, Diarrhea",Treatment of Bacterial infections,NaN,NaN,NaN,NaN,NaN,No,ANTI INFECTIVES,NaN
1,2,Azithral 500 Tablet,132.36,False,Alembic Pharmaceuticals Ltd,allopathy,strip of 5 tablets,Azithromycin (500mg),NaN,Zithrocare 500mg Tablet,...,"Vomiting, Nausea, Abdominal pain, Diarrhea",Treatment of Bacterial infections,NaN,NaN,NaN,NaN,Macrolides,No,ANTI INFECTIVES,Macrolides
2,3,Ascoril LS Syrup,118.00,False,Glenmark Pharmaceuticals Ltd,allopathy,bottle of 100 ml Syrup,Ambroxol (30mg/5ml),Levosalbutamol (1mg/5ml),Solvin LS Syrup,...,"Nausea, Vomiting, Diarrhea, Upset stomach, Sto...",Treatment of Cough with mucus,NaN,NaN,NaN,NaN,NaN,No,RESPIRATORY,NaN
3,4,Allegra 120mg Tablet,218.81,False,Sanofi India Ltd,allopathy,strip of 10 tablets,Fexofenadine (120mg),NaN,Lcfex Tablet,...,"Headache, Drowsiness, Dizziness, Nausea",Treatment of Sneezing and runny nose due to al...,Treatment of Allergic conditions,NaN,NaN,NaN,Diphenylmethane Derivative,No,RESPIRATORY,H1 Antihistaminics (second Generation)
4,4,Allegra 120mg Tablet,218.81,False,Sanofi India Ltd,allopathy,strip of 10 tablets,Fexofenadine (120mg),NaN,Lcfex Tablet,...,"Headache, Drowsiness, Dizziness, Nausea",Treatment of Sneezing and runny nose due to al...,Treatment of Allergic conditions,NaN,NaN,NaN,Diphenylmethane Derivative,No,RESPIRATORY,H1 Antihistaminics (second Generation)


In [15]:
# =========================
# RENAME COLUMNS
# =========================

df = df.rename(columns={
    'name': 'brand_name',
    'price(₹)': 'price',
    'manufacturer_name': 'manufacturer',
    'short_composition1': 'comp1',
    'short_composition2': 'comp2',
    'Therapeutic Class': 'therapeutic_class'
})


In [11]:
# =========================
# KEEP ONLY REQUIRED
# =========================

df = df[['brand_name', 'price', 'manufacturer', 'comp1', 'comp2', 'therapeutic_class',
         'substitute0','substitute1','substitute2','substitute3','substitute4']]

In [12]:
# =========================
# CLEAN DATA
# =========================

df = df.dropna(subset=['brand_name', 'comp1'])

df['price'] = pd.to_numeric(df['price'], errors='coerce')
df = df.dropna(subset=['price'])

df = df.drop_duplicates()

print("After cleaning:", df.shape)


After cleaning: (256359, 11)


In [13]:
# =========================
# CREATE SALT COLUMN
# =========================

df['salt_clean'] = (
    df['comp1'].fillna('') + " " + df['comp2'].fillna('')
).str.lower()

df['salt_clean'] = df['salt_clean'].str.replace(r'[^a-z0-9 ]', '', regex=True)
df['salt_clean'] = df['salt_clean'].str.replace(r'\d+mg', '', regex=True)
df['salt_clean'] = df['salt_clean'].str.strip()

print(df[['comp1','comp2','salt_clean']].head())


                   comp1                       comp2  \
0  Amoxycillin  (500mg)      Clavulanic Acid (125mg)   
1   Azithromycin (500mg)                         NaN   
2   Ambroxol (30mg/5ml)    Levosalbutamol (1mg/5ml)    
3   Fexofenadine (120mg)                         NaN   
4   Fexofenadine (120mg)                         NaN   

                          salt_clean  
0   amoxycillin      clavulanic acid  
1                       azithromycin  
2  ambroxol 5ml   levosalbutamol 5ml  
3                       fexofenadine  
4                       fexofenadine  


In [17]:
# =========================
# REDUCE SIZE (OPTIONAL)
# =========================

df = df.sample(min(50000, len(df)), random_state=42)

In [19]:
# =========================
# SAVE CLEAN DATA
# =========================

df.to_csv("medicines_clean.csv", index=False)

In [20]:
# =========================
# CORE ENGINE
# =========================

def get_alternatives(medicine_name):
    med = df[df['brand_name'].str.lower() == medicine_name.lower()]
    
    if med.empty:
        print("Medicine not found")
        return None
    
    med = med.iloc[0]
    
    salt = med['salt_clean']
    original_price = med['price']
    
    same_salt = df[df['salt_clean'] == salt].copy()
    same_salt = same_salt.sort_values(by='price')
    
    same_salt['savings_percent'] = ((original_price - same_salt['price']) / original_price) * 100
    
    return same_salt[['brand_name','price','manufacturer','savings_percent']].head(5)



In [21]:
# =========================
# TEST
# =========================

print("\nTEST RESULT:\n")
result = get_alternatives("Augmentin 625 Duo Tablet")
print(result)


TEST RESULT:

Medicine not found
None
